# 03. Domain Generalization: IRM, GroupDRO, V-REx, Deep CORAL, and DANN

Empirical Risk Minimization (ERM) assumes training and deployment data are drawn from identical distributions. When environmental artifacts or demographic variations introduce distribution shifts, ERM models latch onto spurious correlations that collapse out-of-distribution. Domain generalization algorithms enforce invariant causal representations across environments.

### Core Generalization Pillars
1. **Invariant Risk Minimization (IRMv1)**: Enforces that the optimal linear classifier on top of representation $\Phi$ is stationary across all training environments.
2. **Variance Risk Extrapolation (V-REx)**: Penalizes the variance of empirical risks across environments to enforce robust generalization beyond the training convex hull.
3. **Group Distributionally Robust Optimization (GroupDRO)**: Minimizes worst-case risk across predefined environmental groups via online exponentiated gradient updates.
4. **Deep CORAL**: Aligns second-order covariance statistics between domains without adversarial instability.


## Mathematical Foundations & Governing Equations

### 1. IRMv1 Stationarity Penalty
$$\nabla_{w=1.0} R^e(w \cdot \Phi) = \frac{1}{N_e} \sum_{i=1}^{N_e} z_i^e (\sigma(z_i^e) - y_i^e), \quad \text{Penalty} = \|\nabla_{w=1.0} R^e\|_2^2$$

### 2. V-REx Cross-Environment Risk Variance
$$\text{Var}_e(\mathcal{L}_e) = \frac{1}{|\mathcal{E}|} \sum_{e \in \mathcal{E}} (\mathcal{L}_e - \bar{\mathcal{L}})^2$$

### 3. GroupDRO Exponentiated Gradient Weights
$$\mathcal{L}_{\text{DRO}} = \sum_{e} q_e \mathcal{L}_e, \quad q_e^{(t+1)} \propto q_e^{(t)} \exp(\eta \mathcal{L}_e^{(t)})$$

### 4. Deep CORAL Covariance Alignment
$$\mathcal{L}_{\text{CORAL}} = \frac{1}{4 d^2} \|C_s - C_t\|_F^2$$


In [ ]:
# Setup, imports, and golden weight loading
import sys
from pathlib import Path
import torch
from safetensors.torch import load_file

repo_root = Path.cwd()
while not (repo_root / "Cargo.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

weights_path = repo_root / "weights" / "09_causal_ai_and_evolutionary_search" / "03_domain_generalization_irm_groupdro_vrex_coral_and_dann.safetensors"
assert weights_path.exists(), f"Weights file not found: {weights_path}"
weights = load_file(str(weights_path))
print(f"Loaded {len(weights)} golden tensors on cpu")


In [ ]:
def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return x @ weight + bias

def irm_v1_penalty(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    probs = torch.sigmoid(logits)
    grad = torch.mean(logits * (probs - targets))
    return (grad * grad).unsqueeze(0)

def vrex_variance_penalty(env_losses: torch.Tensor) -> torch.Tensor:
    diff = env_losses - torch.mean(env_losses)
    return torch.mean(diff * diff).unsqueeze(0)

def group_dro_update(env_losses: torch.Tensor, q_weights: torch.Tensor, eta: float = 0.1):
    dro_loss = torch.sum(q_weights * env_losses).unsqueeze(0)
    unnorm_q = q_weights * torch.exp(eta * env_losses)
    return dro_loss, unnorm_q / torch.sum(unnorm_q)

def deep_coral_loss(phi_s: torch.Tensor, phi_t: torch.Tensor) -> torch.Tensor:
    ns, d = phi_s.shape
    nt, _ = phi_t.shape
    cent_s = phi_s - torch.mean(phi_s, dim=0, keepdim=True)
    cent_t = phi_t - torch.mean(phi_t, dim=0, keepdim=True)
    cov_s = (cent_s.t() @ cent_s) / float(ns - 1)
    cov_t = (cent_t.t() @ cent_t) / float(nt - 1)
    diff = cov_s - cov_t
    return (torch.sum(diff * diff) / (4.0 * float(d * d))).unsqueeze(0)


In [ ]:
# Multi-track numerical parity assertions
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
err_anchor = float(torch.max(torch.abs(y_anchor - weights["test_output"])))
assert err_anchor < 1e-5, f"Anchor parity error: {err_anchor:.2e}"

irm_pen = irm_v1_penalty(weights["irm_logits"], weights["irm_targets"])
err_irm = float(torch.max(torch.abs(irm_pen - weights["irm_penalty_ref"])))
assert err_irm < 1e-5, f"IRM penalty parity error: {err_irm:.2e}"

vrex = vrex_variance_penalty(weights["env_losses"])
err_vrex = float(torch.max(torch.abs(vrex - weights["vrex_penalty_ref"])))
assert err_vrex < 1e-5, f"V-REx parity error: {err_vrex:.2e}"

dro_loss, q_up = group_dro_update(weights["env_losses"], weights["q_weights"], eta=0.1)
err_dro = float(torch.max(torch.abs(dro_loss - weights["dro_loss_ref"])))
err_qup = float(torch.max(torch.abs(q_up - weights["q_updated_ref"])))
assert max(err_dro, err_qup) < 1e-5, f"GroupDRO parity error"

coral = deep_coral_loss(weights["phi_source"], weights["phi_target"])
err_coral = float(torch.max(torch.abs(coral - weights["coral_loss_ref"])))
assert err_coral < 1e-5, f"Deep CORAL parity error: {err_coral:.2e}"

print(f"All parity assertions passed successfully (max error {max(err_anchor, err_irm, err_vrex, err_dro, err_qup, err_coral):.2e})")


In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table
b_irm = benchmark_fn(lambda: irm_v1_penalty(weights["irm_logits"], weights["irm_targets"]), iters=100, name="IRMv1 Stationarity Penalty")
b_vrex = benchmark_fn(lambda: vrex_variance_penalty(weights["env_losses"]), iters=100, name="V-REx Environment Variance")
b_dro = benchmark_fn(lambda: group_dro_update(weights["env_losses"], weights["q_weights"]), iters=100, name="GroupDRO Exponentiated Step")
b_coral = benchmark_fn(lambda: deep_coral_loss(weights["phi_source"], weights["phi_target"]), iters=100, name="Deep CORAL Covariance Loss")
print(format_benchmark_table([b_irm, b_vrex, b_dro, b_coral]))


In [ ]:
# Failure Mode Contrast: ERM Spurious Shortcut vs. Invariant Risk Minimization (IRM)
# Setting: True causal feature X generates label Y = Bernoulli(sigmoid(2.0 * X)) across all domains
# Spurious color feature C has inverted correlation across domains (Env 1: C = Y; Env 2: C = 1 - Y)
torch.manual_seed(42)
n = 1000
x_e1, x_e2 = torch.randn(n, 1), torch.randn(n, 1)
y_e1 = torch.bernoulli(torch.sigmoid(2.0 * x_e1))
y_e2 = torch.bernoulli(torch.sigmoid(2.0 * x_e2))

# Invariant causal model (logits = 2.0 * X) satisfies stationarity across both environments
pen_causal = (irm_v1_penalty(2.0 * x_e1, y_e1) + irm_v1_penalty(2.0 * x_e2, y_e2)).item()

# Spurious model relying on color: logits = 4.0 * (C - 0.5)
c_e1, c_e2 = y_e1.clone(), 1.0 - y_e2.clone()
pen_spurious = (irm_v1_penalty(4.0 * (c_e1 - 0.5), y_e1) + irm_v1_penalty(4.0 * (c_e2 - 0.5), y_e2)).item()

print(f"IRM Penalty on Invariant Causal Feature: {pen_causal:.6f} (Stationary optimal across domains!)")
print(f"IRM Penalty on Spurious Color Shortcut:  {pen_spurious:.6f} (Severely penalized by IRM!)")

assert pen_causal < 0.01, "Causal feature must have near-zero gradient stationarity penalty"
assert pen_spurious > 1.0, "Spurious feature must incur massive cross-domain gradient penalty"
print("Pathological failure mode contrast successfully verified")


## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Stationarity as Invariance**: Invariant Risk Minimization replaces data augmentation heuristics with a rigorous mathematical condition: optimal linear classification heads must be simultaneously stationary across environments.
2. **Risk Variance Penalization**: V-REx prevents high-frequency environments from dominating optimization by penalizing the cross-environment empirical variance.
3. **Minimax Protection**: GroupDRO adaptively reweights environments via exponentiated gradient ascent to guard minority groups.

---
### Cross-Domain Generalisation Matrix
- **Multi-Site Clinical Imaging**: Eliminates scanner hardware hospital batch effects.
- **Autonomous Vehicle Perception**: Maintains detection accuracy across weather domains (rain, fog, night).
- **Algorithmic Fairness in Credit Scoring**: GroupDRO prevents demographic minority underperformance.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [03_domain_generalization_irm_groupdro_vrex_coral_and_dann.md](../../markdown/09_causal_ai_and_evolutionary_search/03_domain_generalization_irm_groupdro_vrex_coral_and_dann.md)
- **PyTorch 2.x Module:** [03_domain_generalization_irm_groupdro_vrex_coral_and_dann.py](../../pytorch2/09_causal_ai_and_evolutionary_search/03_domain_generalization_irm_groupdro_vrex_coral_and_dann.py)
- **Functional JAX Module:** [03_domain_generalization_irm_groupdro_vrex_coral_and_dann.py](../../jax/09_causal_ai_and_evolutionary_search/03_domain_generalization_irm_groupdro_vrex_coral_and_dann.py)
- **Native Rust Candle Track:** [03_domain_generalization_irm_groupdro_vrex_coral_and_dann.rs](../../candle/09_causal_ai_and_evolutionary_search/03_domain_generalization_irm_groupdro_vrex_coral_and_dann.rs)
- **Hardware Kernel (CPU / WGSL):** [03_domain_generalization_irm_groupdro_vrex_coral_and_dann.rs](../../kernels/09_causal_ai_and_evolutionary_search/03_domain_generalization_irm_groupdro_vrex_coral_and_dann.rs)
- **Golden SafeTensors:** [03_domain_generalization_irm_groupdro_vrex_coral_and_dann.safetensors](../../weights/09_causal_ai_and_evolutionary_search/03_domain_generalization_irm_groupdro_vrex_coral_and_dann.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 02. Causal Treatment Effects](02_causal_treatment_effects_tarnet_cfrnet_dragonnet_dml_and_r_learner.ipynb) | [Master Index](../../README.md#09_causal_ai_and_evolutionary_search) | [Next: 04. Evolution Strategies](04_evolution_strategies_cma_es_openai_es_and_pso.ipynb) &rarr;
